# TEMPLATE 5: TEXT FEATURE ENGINEERING FOR TABULAR DATASETS
### GEMASTIK DATA MINING & HYBRID MACHINE LEARNING PIPELINE
Seringkali di lomba Data Mining, dataset utama bertipe **Tabular** (seperti data perbankan, e-commerce, asuransi, medis), namun memiliki **1 atau 2 kolom teks deskripsi/keluhan**.

Template ini mendemonstrasikan cara mengubah kolom teks menjadi **fitur angka berkepadatan tinggi (Dense Feature Primitives)** untuk digabungkan dengan tabel utama dan dilatih dengan LightGBM/CatBoost.

**Fitur Primitif yang Dihasilkan:**
1. Text Morphometrics: Karakteristik morfologis (panjang teks, rasio huruf kapital, jumlah angka, tanda baca).
2. LSA Semantic Embeddings: TF-IDF + TruncatedSVD (10-30 komponen topik padat).
3. Lexicon Sentimen: Estimasi polaritas nada teks.
4. Penggabungan otomatis ke DataFrame tabular utama untuk boosting performa model tabular.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

SEED = 2026
print("Modul Hybrid Text-Tabular Feature Engineering siap.")


## 1. Memuat Dataset Tabular yang Memiliki Kolom Teks
Sebagai contoh: Data nasabah pengajuan pinjaman/asuransi yang memuat kolom angka (usia, pendapatan) dan kolom teks deskripsi (alasan pinjaman).


In [ ]:
print("Membuat simulasi dataset tabular hibrida...")
sample_hybrid = {
    "customer_id": range(1, 11),
    "age": [25, 42, 33, 50, 29, 38, 45, 23, 60, 31],
    "income_million": [8.5, 25.0, 12.0, 4.5, 9.0, 18.0, 6.0, 5.0, 30.0, 11.5],
    "loan_amount_million": [20.0, 100.0, 40.0, 15.0, 25.0, 70.0, 30.0, 10.0, 150.0, 35.0],
    "loan_purpose_text": [
        "Modal tambahan untuk ekspansi usaha kedai kopi dan beli mesin espresso baru.",
        "Renovasi atap rumah yang bocor parah sebelum musim hujan datang.",
        "Biaya persalinan anak pertama di rumah sakit swasta dan kebutuhan bayi.",
        "Butuh dana darurat untuk melunasi utang pinjol yang sudah jatuh tempo bunga tinggi.",
        "Beli laptop baru untuk menunjang pekerjaan freelance desain grafis profesional.",
        "Investasi pembelian tanah kavling produktif di daerah pinggiran kota.",
        "Biaya pengobatan darurat orang tua yang sedang dirawat intensif di rumah sakit.",
        "Kebutuhan konsumtif beli motor baru untuk operasional harian kerja ojek online.",
        "Ekspansi bisnis restoran keluarga membuka cabang baru di pusat perbelanjaan.",
        "Modal perbaikan mobil operasional toko kelontong yang turun mesin rusak parah."
    ],
    "default_target": [0, 0, 0, 1, 0, 0, 1, 0, 0, 0]  # 1 = Gagal Bayar, 0 = Lancar
}
df = pd.DataFrame(sample_hybrid)
print(df[["customer_id", "income_million", "loan_purpose_text", "default_target"]].head(4))


## 2. Ekstraksi Primitif 1: Fitur Stilometri & Morfologi Teks


In [ ]:
def extract_stylometrics(text_series):
    feats = pd.DataFrame(index=text_series.index)
    feats["text_char_len"] = text_series.astype(str).str.len()
    feats["text_word_count"] = text_series.astype(str).apply(lambda s: len(s.split()))
    feats["text_avg_word_len"] = feats["text_char_len"] / (feats["text_word_count"] + 1)
    feats["text_caps_ratio"] = text_series.astype(str).apply(lambda s: sum(1 for c in s if c.isupper())) / (feats["text_char_len"] + 1)
    feats["text_digit_count"] = text_series.astype(str).apply(lambda s: sum(1 for c in s if c.isdigit()))
    feats["text_punct_count"] = text_series.astype(str).apply(lambda s: len(re.findall(r"[^\w\s]", s)))
    
    # Kata kunci risiko tinggi (financial distress lexicon)
    risk_words = ["pinjol", "utang", "darurat", "jatuh tempo", "bunga", "bocor", "rusak", "sakit"]
    feats["text_risk_kw_count"] = text_series.astype(str).apply(lambda s: sum(1 for kw in risk_words if kw in s.lower()))
    return feats

style_df = extract_stylometrics(df["loan_purpose_text"])
print("Fitur Morfologi Teks:")
print(style_df.head(3))


## 3. Ekstraksi Primitif 2: TF-IDF + Latent Semantic Analysis (LSA 5-10 Komponen)


In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=1000, sublinear_tf=True)
X_tfidf = tfidf.fit_transform(df["loan_purpose_text"].str.lower())

n_lsa = min(5, X_tfidf.shape[1] - 1, len(df) - 1)
svd = TruncatedSVD(n_components=n_lsa, random_state=SEED)
X_lsa = svd.fit_transform(X_tfidf)

lsa_cols = [f"lsa_topic_{i+1}" for i in range(n_lsa)]
lsa_df = pd.DataFrame(X_lsa, columns=lsa_cols, index=df.index)
print("Komponen Semantik LSA:")
print(lsa_df.head(3))


## 4. Penggabungan Fitur Teks ke Matriks Tabular Utama


In [ ]:
# Menggabungkan fitur tabular asli + fitur teks yang baru dibuat
df_enriched = pd.concat([
    df.drop(columns=["loan_purpose_text"]),
    style_df,
    lsa_df
], axis=1)

print(f"Data Awal    : {df.shape[1]} kolom")
print(f"Data Diperkaya: {df_enriched.shape[1]} kolom")
print(f"Daftar Kolom Baru:\n{list(df_enriched.columns)}")


## 5. Pelatihan Model LightGBM pada Data Hibrida


In [ ]:
feature_cols = [c for c in df_enriched.columns if c not in ["customer_id", "default_target"]]
X = df_enriched[feature_cols]
y = df_enriched["default_target"]

params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 8,
    "min_child_samples": 2,
    "random_state": SEED,
    "verbose": -1
}

train_data = lgb.Dataset(X, label=y)
model = lgb.train(params, train_data, num_boost_round=50)

# Feature Importance
importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": model.feature_importance(importance_type="gain")
}).sort_values(by="Importance", ascending=False)

print("=== 10 FITUR PALING PENGARUH TERHADAP PREDIKSI ===")
print(importance.head(10))
